# EPANG-Gen: Introduction

This notebook provides a basic introduction to using EPANG-Gen for training physics-informed neural networks.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from epang_gen import EPANGGen, BayesianPASA, BayesianPINN, set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Define a Simple PDE: Poisson 1D

In [ ]:
def poisson_1d_loss(model, x_colloc, x_bc, u_bc):
    x_colloc.requires_grad_(True)
    u = model(x_colloc)
    u_x = torch.autograd.grad(u, x_colloc, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x_colloc, torch.ones_like(u_x), create_graph=True)[0]
    f = torch.pi**2 * torch.sin(torch.pi * x_colloc)
    loss_pde = torch.mean((u_xx + f)**2)
    u_bc_pred = model(x_bc)
    loss_bc = torch.mean((u_bc_pred - u_bc)**2)
    return loss_pde + loss_bc

def generate_poisson_1d(n_colloc=1000):
    x_colloc = torch.rand(n_colloc, 1) * 2 - 1
    x_bc = torch.tensor([[-1.0], [1.0]])
    u_bc = torch.tensor([[0.0], [0.0]])
    return x_colloc, x_bc, u_bc

## 2. Create Model and Optimizer

In [ ]:
layers = [1, 50, 50, 1]
model = BayesianPINN(layers).to(device)

# Create EPANG-Gen optimizer with adaptive rank selection
pasa = BayesianPASA(initial_rank=10, max_rank=20)
optimizer = EPANGGen(
    model.parameters(),
    lr=1e-3,
    rank=10,
    eigen_update_freq=100,
    pasa=pasa
)

## 3. Training Loop

In [ ]:
data = generate_poisson_1d()
data = [d.to(device) for d in data]

losses = []

for epoch in range(2000):
    def closure():
        optimizer.zero_grad()
        loss = poisson_1d_loss(model, *data)
        loss.backward()
        return loss.item()
    
    loss = optimizer.step(closure)
    losses.append(loss)
    
    if epoch % 200 == 0:
        print(f"Epoch {epoch}: loss = {loss:.6f}")

## 4. Plot Convergence

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Poisson 1D Convergence with EPANG-Gen')
plt.grid(True, alpha=0.3)
plt.show()